# 01 — Delta Lake on MinIO

Create a Delta table on `s3a://spark-warehouse/delta/people/`, mutate it, then time-travel back.

In [ ]:
from spark_session import get_spark
from pyspark.sql import functions as F

spark = get_spark("01-delta")
TABLE = "s3a://spark-warehouse/delta/people"

In [ ]:
people = spark.createDataFrame(
    [(1, "Ada", 36), (2, "Linus", 54), (3, "Grace", 85), (4, "Dennis", 70)],
    ["id", "name", "age"],
)
people.write.format("delta").mode("overwrite").save(TABLE)
spark.read.format("delta").load(TABLE).show()

In [ ]:
# Register a SQL view so we can use UPDATE / DELETE.
spark.sql(f"CREATE OR REPLACE TEMP VIEW people AS SELECT * FROM delta.`{TABLE}`")
spark.sql(f"UPDATE delta.`{TABLE}` SET age = age + 1 WHERE name = 'Ada'")
spark.sql(f"DELETE FROM delta.`{TABLE}` WHERE name = 'Dennis'")
spark.read.format("delta").load(TABLE).orderBy("id").show()

In [ ]:
# Full history — should be at least 3 versions: write, update, delete.
spark.sql(f"DESCRIBE HISTORY delta.`{TABLE}`").select("version", "timestamp", "operation").show(truncate=False)

In [ ]:
# Time-travel: read version 0 (the original insert).
(spark.read.format("delta").option("versionAsOf", 0).load(TABLE).orderBy("id").show())

Inspect the underlying objects in the MinIO console at http://localhost:9001 → bucket `spark-warehouse` → `delta/people/_delta_log/`.

In [ ]:
# Release the SparkSession so cluster cores free up and the History Server can ingest this app.
spark.stop()